<a href="https://colab.research.google.com/github/yasuhisasugiura-crypto/newfile_generator/blob/main/lora_train_filal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [step0.1 setup-install]  ←  このセルの名前（以降のセルも1行目に名前を記載）
VLLM_WHEEL = "https://github.com/vllm-project/vllm/releases/download/v0.26.0/vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
!printf "torch==2.11.0\ntorchvision==0.26.0\n" > /tmp/constraints.txt
!pip install -q "$VLLM_WHEEL" "sudachipy==0.6.11" "sudachidict-core==20260428" -c /tmp/constraints.txt > /tmp/pip_install.log 2>&1
print("exit code:", _exit_code)   # 直前の pip install の終了コード（0 なら成功）
!pip uninstall -y -q torchcodec
!tail -3 /tmp/pip_install.log

exit code: 0


In [ ]:
!pip install -q peft -c /tmp/constraints.txt
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 【学習前の環境確認セル】再起動の後、Drive マウントの後に実行する。読むだけ。
import importlib.metadata as md, platform
def ver(n):
    try:
        return md.version(n)
    except md.PackageNotFoundError:
        return None
print("python", platform.python_version(), "（想定 3.12.x）")
ng = []
for name, want in (("torch", "2.11.0+cu128"), ("transformers", "5.13.1"), ("peft", None)):
    got = ver(name)
    bad = got is None or (want is not None and got != want)
    print("  %-14s いま %-16s 想定 %-16s %s" % (name, got, want or "（入っていればよい）", "NG" if bad else "OK"))
    if bad:
        ng.append(name)
print("  torchao        いま %s（None が望ましい。入っていると、前回のように import エラーになることがある）" % ver("torchao"))
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし", "（L4 が想定）")
print("→ " + ("NG あり: %s。step0.1 / peft のセルからやり直す" % ng if ng else "すべて OK"))

python 3.13.16 （想定 3.12.x）
  torch          いま 2.11.0           想定 2.11.0+cu128     NG
  transformers   いま 5.22.1           想定 5.13.1           NG
  peft           いま 0.21.2           想定 （入っていればよい）       OK
  torchao        いま None（None が望ましい。入っていると、前回のように import エラーになることがある）
GPU: NVIDIA L4 （L4 が想定）
→ NG あり: ['torch', 'transformers']。step0.1 / peft のセルからやり直す


In [ ]:
# 【学習データ確認セル】学習セルを実行する前に実行する。読むだけ（質問文・回答は表示しない）
import hashlib, json, os, time
D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
ts = lambda p: time.strftime("%m/%d %H:%M UTC", time.gmtime(os.stat(p).st_mtime))
if not os.path.isdir(D):
    raise SystemExit("LLM講座最終コンペ_lora が見えない。Drive のマウントを確認する")
CP, AP, MP = D + "/train_contexts_1500.jsonl", D + "/train_answers.json", D + "/train_manifest.json"
for p in (CP, AP, MP):
    if not os.path.exists(p):
        raise SystemExit("ファイルが見えない: " + p.replace(D, "…"))
man = json.load(open(MP, encoding="utf-8"))
sc = man.get("spot_check", {})
print("train_contexts_1500.jsonl: %.1f MB・%d 行・更新 %s" % (os.path.getsize(CP) / 1e6, sum(1 for _ in open(CP, encoding="utf-8")), ts(CP)))
print("由来:", man.get("source"), "/ 答えを含む質問:", man.get("usable_in_top5"), "/ 抜き取り再検索:", "合格" if sc.get("ok") else "未実施または不合格",
      "（集合 %s / %s・順位 %s）" % (sc.get("set_match"), sc.get("n"), sc.get("order_match")))
for name, p in (("train_contexts_1500.jsonl", CP), ("train_answers.json", AP)):
    ok = name in man.get("files", {}) and sha(p) == man["files"][name]["sha256"]
    print("  指紋 %-28s %s" % (name, "一致" if ok else "不一致（確認セルからやり直す）"))
print("既存: merged %s / ckpt_final %s / adapter_final %s" % tuple(os.path.isdir(D + "/" + x) for x in ("merged", "ckpt_final", "adapter_final")))
print("  merged が True なら、学習で上書きされる。ckpt_final が True なら、途中から再開する（意図しない古いものなら、Drive で名前を変えてから進む）")

train_contexts_1500.jsonl: 9.8 MB・1500 行・更新 11/08 08:06 UTC
由来: recovered / 答えを含む質問: 1039 / 抜き取り再検索: 合格 （集合 30 / 30・順位 30）
  指紋 train_contexts_1500.jsonl    一致
  指紋 train_answers.json           一致
既存: merged True / ckpt_final True / adapter_final True
  merged が True なら、学習で上書きされる。ckpt_final が True なら、途中から再開する（意図しない古いものなら、Drive で名前を変えてから進む）


In [ ]:
# 【学習セル（本番版）】LoRA で追加学習する。L02（ローカルで +14 ポイント）と同じ設定。途中で止まっても続きから再開できる。
# 前提: セットアップ（step0.1 → peft → 再起動）を済ませ、Drive をマウントし、環境確認が OK になってから実行する。
TEST_N = None        # 動作確認: 40 / 本番: None（TEST_N を入れると merged_test を上書きする。本番では None のまま）
RESUME = True        # True: Drive に途中保存があれば続きから始める（本番のみ）
SAVE_EVERY = 10      # 何ステップごとに途中保存するか（約 17 分ごと）
import gc, hashlib, json, math, os, random, re, time, unicodedata
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from safetensors.torch import load_file

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし"
print("GPU:", gpu)
if not any(x in gpu for x in ("L4", "A100", "H100")):
    raise SystemExit("bf16 で学習するので L4 以上の GPU を選ぶ（T4 は不可）。ランタイムのタイプを変更する")

D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
OUT_TRAIN = D + ("/merged_test" if TEST_N else "/merged")
CK = D + "/ckpt_final"
BASE_MODEL = "Qwen/Qwen3.5-0.8B"
BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"
MAX_LEN = 3072
ACC = 8
LR = 1e-4
YESNO = {"yes": "はい", "no": "いいえ"}

def answer_prompt(question, contexts):   # P2（提出用ノートブックの [step7.1] の answer_prompt と同じ）
    context_text = "\n\n".join("[chunk_id=%d] %s\n%s" % (c["chunk_id"], c["title"], c["text"]) for c in contexts)
    return (
        "質問: " + question + "\n\n"
        "次の資料を根拠に、上の質問に短く答えてください。\n"
        "答え方のきまり:\n"
        "- answer には、質問の答えになる語句だけを書く（人名・地名・作品名・数値・日付など）。\n"
        "- 文にしない。「〜です」「〜である」を付けない。説明・補足・括弧書き・読み仮名・英語表記を付けない。\n"
        "- 「どちら」「どっち」と聞かれたら、選んだほうの名前だけを書く。\n"
        "- 「はい」か「いいえ」で答える質問には、「はい」か「いいえ」だけを書く。\n"
        "- 資料のどこにも答えが書かれていない場合だけ、answer を「資料からは特定できません」、chunk_id を null にする。\n"
        "chunk_id には、答えが書かれている資料の chunk_id(整数)を1つだけ書く。\n"
        '形式: {"answer":"答えの語句","chunk_id":整数}\n\n'
        "資料:\n" + context_text + "\n\n質問（再掲）: " + question
    )

# 提出用ノートブックでも同じ式を実行し、同じ値になることを確認する（プロンプト一致の確認）
print("プロンプト md5:", hashlib.md5(answer_prompt("Q", [{"chunk_id": 1, "title": "T", "text": "X"}]).encode("utf-8")).hexdigest())

def _norm(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", s or "").lower())

def train_loop(examples, acc, step0, fwd_bwd, opt_step, on_step):
    # step0 ステップぶんを飛ばして続きから学習する（step0=0 なら最初から）。戻り値: 最終ステップ数
    step, skip = step0, step0 * acc
    for i, ex in enumerate(examples, 1):
        if i <= skip:
            continue
        fwd_bwd(ex)
        if i % acc == 0 or i == len(examples):
            opt_step()
            step += 1
            on_step(step)
    return step

random.seed(0); torch.manual_seed(0)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REV)
gold = json.load(open(D + "/train_answers.json", encoding="utf-8"))
items = [json.loads(l) for l in open(D + "/train_contexts_1500.jsonl", encoding="utf-8") if l.strip()]
print("検索済みの質問", len(items), "問")
if not TEST_N and len(items) < 1400:
    raise SystemExit("検索結果が %d 行しかない（1500 のはず）。本番の学習を止める" % len(items))
if not TEST_N:                                   # 確認済みのファイルかを、指紋で照合する
    man_all = json.load(open(D + "/train_manifest.json", encoding="utf-8"))
    man = man_all["files"]
    for name, path in (("train_contexts_1500.jsonl", D + "/train_contexts_1500.jsonl"), ("train_answers.json", D + "/train_answers.json")):
        if name not in man or hashlib.sha256(open(path, "rb").read()).hexdigest() != man[name]["sha256"]:
            raise SystemExit(name + " が、確認済みのものと違う（または未確認）。確認セルからやり直す")
    if man_all.get("source") == "recovered" and not man_all.get("spot_check", {}).get("ok"):
        raise SystemExit("見つけた旧ファイルは、抜き取り再検索（現行の検索との照合）が未実施か不合格。学習は始めない")
    print("指紋の照合: OK（確認済みのファイルで学習する）/ 由来:", man_all.get("source", "rebuilt"))
examples, skipped = [], 0
for it in items:
    ctx, ans = it["contexts"][:5], gold[it["q_id"]]["answers"]
    target = None
    if all(_norm(a) in YESNO for a in ans):
        target = {"answer": YESNO[_norm(ans[0])], "chunk_id": ctx[0]["chunk_id"]}
    else:
        for c in ctx:
            hit = next((a for a in ans if _norm(a) and _norm(a) in _norm(c["title"] + c["text"])), None)
            if hit:
                target = {"answer": hit, "chunk_id": c["chunk_id"]}
                break
    if target is None:
        skipped += 1
        continue
    p = tok.apply_chat_template([{"role": "user", "content": answer_prompt(it["question"], ctx)}],
                                tokenize=False, add_generation_prompt=True, enable_thinking=False)
    p_ids = tok(p, add_special_tokens=False).input_ids
    t_ids = tok(json.dumps(target, ensure_ascii=False, separators=(",", ":")) + "<|im_end|>",
                add_special_tokens=False).input_ids
    if len(p_ids) + len(t_ids) <= MAX_LEN:
        examples.append((p_ids, t_ids))
print("学習に使える", len(examples), "問 / 資料に正解がなく除いた", skipped, "問（前回の Colab テスト: 1038 問 / 461 問）")
random.shuffle(examples)
if TEST_N:
    examples = examples[:TEST_N]
    print("【短時間テスト】", len(examples), "問だけ学習する")

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=BASE_REV, dtype=torch.bfloat16).to("cuda")
model.gradient_checkpointing_enable(); model.enable_input_require_grads(); model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
steps = math.ceil(len(examples) / ACC)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, steps // 10)) * max(0.0, 1 - s / steps))
print("設定: r=16 alpha=32 lr=%g 勾配をためる問数=%d 総ステップ=%d MAX_LEN=%d" % (LR, ACC, steps, MAX_LEN))

SIG = {"n": len(examples), "acc": ACC, "lr": LR, "steps": steps, "max_len": MAX_LEN}   # 途中保存が「今の設定のもの」か確かめる印
step0 = 0
if RESUME and not TEST_N and os.path.exists(CK + "/state.pt"):
    st = torch.load(CK + "/state.pt", map_location="cuda")
    if st.get("sig") != SIG:
        raise SystemExit("途中保存の設定が今の設定と違う（保存: %s / 今: %s）。古い ckpt_final を別名に移してから、もう一度実行する" % (st.get("sig"), SIG))
    set_peft_model_state_dict(model, load_file(CK + "/adapter_model.safetensors"))
    opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); step0 = st["step"]
    print("途中から再開: %d / %d ステップから" % (step0, steps))

def save_ckpt(step):
    try:
        os.makedirs(CK, exist_ok=True)
        model.save_pretrained(CK)
        torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "step": step, "sig": SIG}, CK + "/state.pt.tmp")
        os.replace(CK + "/state.pt.tmp", CK + "/state.pt")
    except Exception as e:                       # 途中保存の失敗で学習を止めない
        print("[警告] 途中保存に失敗（学習は続ける）:", repr(e))

model.train()
t0, acc_loss, acc_n = time.time(), [0.0], [0]
LOG = D + "/train_log_final.jsonl"
oom = [0]

def fwd_bwd(ex):
    p_ids, t_ids = ex
    ids = labels = loss = None
    failed = False
    try:
        ids = torch.tensor([p_ids + t_ids], device="cuda")
        labels = torch.tensor([[-100] * len(p_ids) + t_ids], device="cuda")   # 答えの JSON 部分だけ学習
        loss = model(input_ids=ids, labels=labels).loss / ACC
        loss.backward()
        acc_loss[0] += loss.item() * ACC; acc_n[0] += 1
    except torch.cuda.OutOfMemoryError:                    # メモリ不足の例は飛ばして続ける（ローカル版 v2 と同じ）
        failed = True
    if failed:                                             # 例外の外で解放する（例外の中だと、メモリが残ったままになる）
        ids = labels = loss = None
        oom[0] += 1
        gc.collect(); torch.cuda.empty_cache()
        print("[警告] メモリ不足で1問飛ばした（累計 %d 問）" % oom[0])
        if oom[0] > 20:
            raise RuntimeError("メモリ不足が20問を超えた。止める")

def opt_step():
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    opt.step(); sched.step(); opt.zero_grad(set_to_none=True)

def on_step(step):
    if TEST_N or step % 10 == 0 or step == steps:
        rec = {"step": step, "steps": steps, "loss": round(acc_loss[0] / max(1, acc_n[0]), 4),
               "min": round((time.time() - t0) / 60, 1), "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1)}
        print("step %(step)d/%(steps)d loss %(loss).4f（%(min).1f 分・メモリ最大 %(peak_gb)s GB）" % rec)
        if not TEST_N:
            with open(LOG, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        acc_loss[0], acc_n[0] = 0.0, 0
    if not TEST_N and step % SAVE_EVERY == 0 and step < steps:
        save_ckpt(step)

final_step = train_loop(examples, ACC, step0, fwd_bwd, opt_step, on_step)
print("学習完了: %d ステップ（%.1f 分）/ メモリ不足で飛ばした問題: %d 問" % (final_step, (time.time() - t0) / 60, oom[0]))
if not TEST_N:
    model.save_pretrained(D + "/adapter_final")          # LoRA 単体も残す（再統合・別実験用）
merged = model.merge_and_unload().to(torch.float16)
merged.save_pretrained(OUT_TRAIN)
tok.save_pretrained(OUT_TRAIN)
print("保存した:", OUT_TRAIN)
if not TEST_N and os.path.isdir(CK):                      # 完了したら途中保存を別名にする（次の学習が、古い途中保存から再開してしまうのを防ぐ）
    done = CK + "_done_" + time.strftime("%Y%m%d_%H%M%S")
    os.rename(CK, done)
    print("途中保存を別名にした:", done)

GPU: NVIDIA L4
プロンプト md5: 1e990717ef9ebb51535b233deec1503e
検索済みの質問 1500 問
指紋の照合: OK（確認済みのファイルで学習する）/ 由来: recovered
学習に使える 1038 問 / 資料に正解がなく除いた 461 問（前回の Colab テスト: 1038 問 / 461 問）


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

設定: r=16 alpha=32 lr=0.0001 勾配をためる問数=8 総ステップ=130 MAX_LEN=3072
途中から再開: 20 / 130 ステップから


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


step 30/130 loss 0.1510（2.2 分・メモリ最大 9.0 GB）
step 40/130 loss 0.1119（4.3 分・メモリ最大 9.0 GB）
step 50/130 loss 0.1241（6.5 分・メモリ最大 11.0 GB）
step 60/130 loss 0.1234（8.7 分・メモリ最大 11.0 GB）
step 70/130 loss 0.1108（10.9 分・メモリ最大 11.0 GB）
step 80/130 loss 0.1077（13.1 分・メモリ最大 11.0 GB）
step 90/130 loss 0.1556（15.3 分・メモリ最大 11.0 GB）
step 100/130 loss 0.1721（17.5 分・メモリ最大 11.0 GB）
step 110/130 loss 0.0974（19.7 分・メモリ最大 11.0 GB）
step 120/130 loss 0.1033（21.9 分・メモリ最大 11.0 GB）
step 130/130 loss 0.1076（24.0 分・メモリ最大 11.0 GB）
学習完了: 130 ステップ（24.0 分）/ メモリ不足で飛ばした問題: 0 問


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

保存した: /content/drive/MyDrive/LLM講座最終コンペ_lora/merged
途中保存を別名にした: /content/drive/MyDrive/LLM講座最終コンペ_lora/ckpt_final_done_20261108_132049


In [ ]:
# 【学習の記録セル（任意）】学習した環境と結果を Drive の train_manifest.json に残す（再現・レポート用）
# 学習セルを実行したセッションのまま実行する（学習セルの変数 answer_prompt・examples を使う）。読む・追記するだけで、学習結果には触れない。
import hashlib, importlib.metadata as md, json, os, platform, time
D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
MP, LOG = D + "/train_manifest.json", D + "/train_log_final.jsonl"
def ver(n):
    try:
        return md.version(n)
    except md.PackageNotFoundError:
        return None
recs = []
for line in reversed(open(LOG, encoding="utf-8").read().splitlines()):     # 今回の実行ぶんだけ（step 10 まで遡る）
    r = json.loads(line); recs.append(r)
    if r["step"] == 10:
        break
recs.reverse()
if not recs or recs[0]["step"] != 10 or recs[-1]["step"] != recs[-1]["steps"]:
    raise SystemExit("学習ログが想定の形ではない（step 10〜最終ステップ）。画面を貼る")
md5 = hashlib.md5(answer_prompt("Q", [{"chunk_id": 1, "title": "T", "text": "X"}]).encode("utf-8")).hexdigest() if "answer_prompt" in globals() else None
rec = {"train_env": {"python": platform.python_version(), "torch": ver("torch"), "transformers": ver("transformers"), "peft": ver("peft"),
                     "gpu": __import__("torch").cuda.get_device_name(0)},
       "train_result": {"steps": recs[-1]["steps"], "minutes": recs[-1]["min"], "first_loss": recs[0]["loss"], "last_loss": recs[-1]["loss"],
                        "peak_gb": max(r["peak_gb"] for r in recs), "n_examples": len(examples) if "examples" in globals() else None,
                        "prompt_md5": md5, "lora": {"r": 16, "alpha": 32, "lr": 1e-4, "acc": 8, "epochs": 1}},
       "train_recorded_at_utc": time.strftime("%Y-%m-%d %H:%M", time.gmtime())}
man = json.load(open(MP, encoding="utf-8"))
man.update(rec)
tmp = MP + ".tmp"
json.dump(man, open(tmp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
os.replace(tmp, MP)
print(json.dumps(rec, ensure_ascii=False, indent=1))
print("記録した:", MP.replace(D, "…"), "/ merged:", os.path.isdir(D + "/merged"), "/ adapter_final:", os.path.isdir(D + "/adapter_final"),
      "/ ckpt_final:", os.path.isdir(D + "/ckpt_final"), "/ ckpt_final_done_*:", [n for n in os.listdir(D) if n.startswith("ckpt_final_done_")])

{
 "train_env": {
  "python": "3.13.16",
  "torch": "2.11.0",
  "transformers": "5.22.1",
  "peft": "0.21.2",
  "gpu": "NVIDIA L4"
 },
 "train_result": {
  "steps": 130,
  "minutes": 24.0,
  "first_loss": 0.2908,
  "last_loss": 0.1076,
  "peak_gb": 11.2,
  "n_examples": 1038,
  "prompt_md5": "1e990717ef9ebb51535b233deec1503e",
  "lora": {
   "r": 16,
   "alpha": 32,
   "lr": 0.0001,
   "acc": 8,
   "epochs": 1
  }
 },
 "train_recorded_at_utc": "2026-11-08 13:20"
}
記録した: …/train_manifest.json / merged: True / adapter_final: True / ckpt_final: False / ckpt_final_done_*: ['ckpt_final_done_20261108_123302', 'ckpt_final_done_20261108_132049']
